In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (confusion_matrix,accuracy_score,
recall_score,f1_score,roc_auc_score,precision_score)

df=pd.read_csv(r'C:\Users\fathi\OneDrive\Desktop\5thsem_OJT\project_diabetes\data\processed\risvana_p1w2_features.csv')
feature_cols = [
    'age',
    'pregnancies',
    'glucose',
    'blood_pressure',
    'skin_thickness',
    'insulin',
    'bmi',
    'diabetes_pedigree',
    'bmi category',
    'glucose category'
]
x= pd.get_dummies(df[feature_cols],drop_first=True)
y=df["outcome"]

x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)
scaler = StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

logreg = LogisticRegression(max_iter=1000).fit(x_train_scaled,y_train)
knn = KNeighborsClassifier(n_neighbors=25).fit(x_train_scaled, y_train)
decision_tree = DecisionTreeClassifier(max_depth=3,random_state=42).fit(x_train, y_train)

def score_row(name, pred, scores=None):
    cm = confusion_matrix(y_test, pred)
    tn, fp, fn, tp = cm.ravel()

    r = {
        'model' : name,
        'accuracy' : round(accuracy_score(y_test, pred),4),
        'precision' : round(precision_score(y_test, pred, zero_division=0),4),
        'recall' : round(recall_score(y_test, pred, zero_division=0),4),
        'f1' : round(f1_score(y_test,pred,zero_division=0),4),
        'found' : int(tp), 'missed' : int(fn), 'false_alarms' : int(fp)
    }

    if scores is not None :
        r['auc'] = roc_auc_score(y_test, scores)
    return r


print('train:', x_train.shape, 'test:', x_test.shape)
print('Diabetic patients in the test set:', int(np.sum(y_test)),'of', len(y_test))
for nm, m, Xt in [('Logistic Regression', logreg, x_test_scaled),('Decision Tree (d=3)', decision_tree, x_test),('KNN (k=25)', knn, x_test_scaled)]:
    print(f' {nm:22s} accuracy {accuracy_score(y_test, m.predict(Xt)):.4f}')

train: (760, 13) test: (190, 13)
Diabetic patients in the test set: 55 of 190
 Logistic Regression    accuracy 0.7526
 Decision Tree (d=3)    accuracy 0.7211
 KNN (k=25)             accuracy 0.7474


resate the target you are aiming at

In [30]:
TARGET_RECALL = 0.70
GUARD_F1 = 0.50
BEST_SO_FAR = 0.6182  # Week 5 Logistic Regression at threshold 0.30

print("PROJECT METRIC : recall  (guard: F1 >=", GUARD_F1, ")")
print("TARGET          : recall >=", TARGET_RECALL)
print("BEST SO FAR     :", BEST_SO_FAR, "(Week 5 baseline at threshold 0.30)")
print("GAP TO CLOSE    :", round(TARGET_RECALL - BEST_SO_FAR, 4))

PROJECT METRIC : recall  (guard: F1 >= 0.5 )
TARGET          : recall >= 0.7
BEST SO FAR     : 0.6182 (Week 5 baseline at threshold 0.30)
GAP TO CLOSE    : 0.0818


## Explain How SVM Reaches a Decision

- **SVM** is a supervised learning algorithm mainly used for classification.
- It finds the **best decision boundary (hyperplane)** to separate different classes.
- SVM tries to create the **maximum margin** between the classes.
- **Margin** is the distance between the decision boundary and the closest data points.
- The closest data points are called **Support Vectors**.
- For a new data point, SVM checks which side of the decision boundary it falls on.
- **One side → Class 0**
- **Other side → Class 1**
- SVM uses a **decision function** to calculate the position of the data point.
- **Positive score → one class**, **negative score → another class**.
- **Decision boundary = score 0**.
- SVM can also use **kernel functions** to separate data that cannot be separated by a straight line.

Train an svm with defult settings

In [32]:
from sklearn.svm import SVC
svm_rbf= SVC(random_state=42)
svm_rbf.fit(x_train_scaled,y_train)
pred_rbf = svm_rbf.predict(x_test_scaled)
print("filtted:",svm_rbf)
print("pattient flagged:",int(pred_rbf.sum()),"of",len(pred_rbf))


filtted: SVC(random_state=42)
pattient flagged: 17 of 190


In [33]:
row_rbf = score_row('SVC rbf ', pred_rbf,svm_rbf.decision_function(x_test_scaled))
print(pd.DataFrame([row_rbf]).to_string(index=False))
print()
print('Target recall:', TARGET_RECALL,'| Best so far:', BEST_SO_FAR)
print('This model :', row_rbf['recall'])

   model  accuracy  precision  recall     f1  found  missed  false_alarms     auc
SVC rbf     0.7263     0.5882  0.1818 0.2778     10      45             7 0.64431

Target recall: 0.7 | Best so far: 0.6182
This model : 0.1818
